# 7. HARQ

HARQ는 **HAR 모형에 Q(Quarticity)를 더한 것**이야. 그래서 HAR을 먼저 확실히 이해하고, 그다음 "RV에는 측정오차가 있다"는 문제를 다루고, 마지막에 Q가 그 문제를 어떻게 푸는지 보는 순서로 갈게.

---

## 0단계: 목표

내일의 실현분산 $RV_{t+1}$을 오늘까지의 정보로 예측하는 거야. 변동성은 수익률 방향과 달리 **예측이 꽤 잘 되는** 변수라서, 이 분야에서 가장 성숙한 주제 중 하나야.

---

## 1단계: 변동성의 두 가지 성질

**변동성 군집(volatility clustering).** 변동성이 큰 날 다음에는 큰 날이, 작은 날 다음에는 작은 날이 오는 경향이 있어. 그래서 어제 RV가 내일 RV의 좋은 예측변수가 돼.

**장기기억(long memory).** 그 영향이 하루 이틀이 아니라 **몇 주, 몇 달에 걸쳐 천천히** 사라져. 한 달 전의 폭락장이 아직도 오늘의 변동성에 영향을 주는 식이야.

첫 번째 성질만 있다면 $RV_{t+1} = \beta_0 + \beta_1 RV_t$ 같은 AR(1)이면 충분해. 문제는 두 번째 성질이야. AR(1)은 충격이 **지수적으로 빨리** 사라지는 구조라 장기기억을 못 잡아. 그렇다고 과거 22일을 전부 넣는 AR(22)는 계수가 23개라 추정이 불안정해.

---

## 2단계: HAR 모형 (Corsi 2009)

해결책은 과거를 **일·주·월 세 덩어리의 평균**으로 요약하는 거야.

$$
RV_{t+1} = \beta_0 + \beta_d\,RV_t + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)} + \varepsilon_{t+1}
$$

$$
RV_t^{(w)} = \frac{1}{5}\sum_{j=0}^{4} RV_{t-j}, \qquad RV_t^{(m)} = \frac{1}{22}\sum_{j=0}^{21} RV_{t-j}
$$

- $RV_t$: 어제 하루
- $RV_t^{(w)}$: 지난 5 거래일 평균 (1주)
- $RV_t^{(m)}$: 지난 22 거래일 평균 (1달)

일반 OLS로 추정할 수 있어.

**HAR(Heterogeneous AutoRegressive)이라는 이름의 직관:** 시장에는 투자 시계가 서로 **다른(heterogeneous)** 참여자들이 있다는 가설에서 나왔어. 단타 트레이더는 어제의 변동에 반응하고, 스윙 트레이더는 지난주, 기관은 지난달의 변동에 반응해. 이들의 반응이 합쳐져 오늘의 변동성이 된다는 거야.

---

## 3단계: HAR이 장기기억을 흉내 내는 원리 (유도)

HAR은 사실 **제약이 걸린 AR(22)**야. 주·월 평균을 풀어 쓰면, 각 과거 날짜의 RV에 붙는 계수가 이렇게 돼.

| 과거 날짜 | $RV_{t-j}$에 붙는 계수 |
|---|---|
| j = 0 (어제) | $\beta_d + \dfrac{\beta_w}{5} + \dfrac{\beta_m}{22}$ |
| j = 1 ~ 4 | $\dfrac{\beta_w}{5} + \dfrac{\beta_m}{22}$ |
| j = 5 ~ 21 | $\dfrac{\beta_m}{22}$ |

예를 들어 계수가 $\beta_d = 0.4$, $\beta_w = 0.35$, $\beta_m = 0.2$라고 하자(설명용 숫자야).

- 어제: $0.4 + 0.07 + 0.009 = 0.479$
- 2~5일 전: $0.07 + 0.009 = 0.079$
- 6~22일 전: $0.009$

**계단 모양으로 천천히 줄어드는 가중치**가 나와. 계수 4개만으로 "최근은 크게, 먼 과거도 조금은" 반영하는 장기기억 패턴을 근사하는 거야. 단순하지만 예측력이 좋아서, 이후 거의 모든 변동성 예측 연구의 **기본 벤치마크**가 됐어.

---

## 4단계: 문제 — RV에는 측정오차가 있다

1장에서 "N → ∞이면 RV가 진짜 분산(IV)에 수렴한다"고 했지. 거꾸로 말하면 **N이 유한하면 오차가 있다**는 거야.

$$
RV_t = IV_t + u_t
$$

$u_t$는 측정오차야. 이 오차가 얼마나 큰지 직접 유도해보자. 점프가 없고 하루 동안 σ가 일정하다고 가정하면, 앞에서 쓴 표기 그대로:

$$
RV_t = \sum_{i=1}^{N}\frac{\sigma^2}{N}z_i^2 = \frac{\sigma^2}{N}\sum_{i=1}^{N}z_i^2
$$

분산을 구하면 ($z_i$ 독립):

$$
\text{Var}(RV_t) = \frac{\sigma^4}{N^2}\sum_{i=1}^{N}\text{Var}(z_i^2) = \frac{\sigma^4}{N^2}\cdot N\cdot\text{Var}(z^2)
$$

$\text{Var}(z^2)$는 모멘트로 계산할 수 있어. 표준정규분포의 4차 모멘트(첨도)가 3이었지.

$$
\text{Var}(z^2) = E[z^4] - (E[z^2])^2 = 3 - 1 = 2
$$

따라서:

$$
\boxed{\text{Var}(u_t) = \frac{2\sigma^4}{N}}
$$

두 가지를 읽어낼 수 있어.

- **N이 작을수록** 오차가 커.
- **σ⁴에 비례**해. 변동성이 큰 날일수록 측정오차가 **훨씬** 커져(제곱의 제곱이니까).

---

## 5단계: 10분봉에서 오차가 얼마나 큰가

상대오차(오차 표준편차 ÷ 진짜 값)로 보면:

$$
\frac{\sqrt{2\sigma^4/N}}{\sigma^2} = \sqrt{\frac{2}{N}}
$$

| 봉 단위 | N | 상대오차 $\sqrt{2/N}$ |
|---|---|---|
| **10분** | **39** | **약 23%** |
| 5분 | 78 | 약 16% |
| 1분 | 390 | 약 7% |

**10분봉 RV는 매일 대략 ±23% 정도 흔들린다는 뜻이야.** 이건 정규분포라는 이상적인 가정에서 나온 값이라, 실제로는 이보다 나쁠 가능성이 높아. 네가 앞으로 10분봉 RV를 쓸 때 반드시 기억해야 할 숫자야.

---

## 6단계: 측정오차가 회귀계수를 깎아먹는다 — 감쇠편의 (유도)

설명변수에 오차가 끼면 OLS 계수가 **0 쪽으로 쪼그라들어.** 이걸 감쇠편의(attenuation bias)라고 해.

진짜 관계가 $y = b\,x^* + e$인데, 우리는 $x^*$ 대신 오차가 섞인 $x = x^* + u$만 관측한다고 하자(u는 $x^*$, e와 무관). OLS 기울기는 $\text{Cov}(y, x)/\text{Var}(x)$로 수렴하니까:

$$
\text{Cov}(y, x) = \text{Cov}(b\,x^* + e,\ x^* + u) = b\,\text{Var}(x^*)
$$

$$
\text{Var}(x) = \text{Var}(x^*) + \text{Var}(u)
$$

$$
\hat b \;\to\; b\cdot\underbrace{\frac{\text{Var}(x^*)}{\text{Var}(x^*) + \text{Var}(u)}}_{< 1}
$$

오차 분산이 클수록 이 비율이 작아지고, 계수가 더 쪼그라들어.

**HAR에 적용하면:** 어제 RV의 계수 $\beta_d$가 과소추정돼. 그런데 더 중요한 포인트가 있어. 4단계에서 봤듯 **오차의 크기가 날마다 달라.** 조용한 날의 RV는 비교적 정확하고, 요동친 날의 RV는 부정확해. 그런데 HAR은 **모든 날에 같은 $\beta_d$**를 써. 정확한 날엔 너무 적게 믿고, 부정확한 날엔 너무 많이 믿는 **어정쩡한 타협**인 셈이야.

주·월 평균은 이 문제가 덜해. 5일, 22일을 평균하면서 오차가 서로 상쇄되니까(분산투자 원리와 같아). 그래서 문제는 주로 **일별 항**에서 생겨.

---

## 7단계: 측정오차의 크기를 데이터로 추정하기 — Realized Quarticity

오차 분산은 $2\sigma^4/N$이니, **σ⁴만 추정하면** 날마다 오차 크기를 알 수 있어. RV가 σ²를 추정했듯, 4제곱으로 σ⁴를 추정하는 거야.

$$
RQ_t = \frac{N}{3}\sum_{i=1}^{N} r_{t,i}^4
$$

**N/3은 왜 붙나 (유도):** 봉 하나의 4제곱 기댓값은

$$
E[r_i^4] = \left(\frac{\sigma}{\sqrt{N}}\right)^4 E[z^4] = \frac{\sigma^4}{N^2}\cdot 3
$$

N개를 합하면 $N \cdot \frac{3\sigma^4}{N^2} = \frac{3\sigma^4}{N}$이야. 여기에 $N/3$을 곱하면 정확히 $\sigma^4$가 돼. 여기서 3은 정규분포의 첨도야. BV에서 π/2가 보정상수였던 것과 같은 역할이지.

그러면 날마다의 측정오차 분산은 이렇게 추정돼.

$$
\widehat{\text{Var}}(u_t) = \frac{2}{N}RQ_t
$$

"quarticity"는 4제곱(quartic)에서 온 이름이고, 모멘트 설명 때 예고했던 그 4차 모멘트야.

---

## 8단계: HARQ 모형 (Bollerslev, Patton, Quaedvlieg 2016)

아이디어는 한 문장이야. **어제 RV의 계수를, 어제 RV가 얼마나 정확했는지에 따라 날마다 다르게 하자.**

$$
\beta_{d,t} = \beta_d + \beta_{dQ}\sqrt{RQ_t}
$$

이걸 HAR에 넣으면:

$$
\boxed{RV_{t+1} = \beta_0 + \big(\beta_d + \beta_{dQ}\sqrt{RQ_t}\big)RV_t + \beta_w RV_t^{(w)} + \beta_m RV_t^{(m)} + \varepsilon_{t+1}}
$$

회귀식으로 보면 설명변수 $\sqrt{RQ_t}\cdot RV_t$ **하나만 추가**된 거라, 여전히 OLS로 추정돼.

**기대 부호: $\beta_{dQ} < 0$.** RQ가 큰 날(측정이 부정확한 날)에는 어제 RV의 가중치를 **낮추고**, RQ가 작은 날(정확한 날)에는 **높이는** 거야. 6단계의 감쇠편의 논리를 날마다 적용하는 셈이지.

**숫자 예시 (설명용 가상 값):** $\beta_d = 0.5$, $\beta_{dQ} = -0.1$이라면

| 날 | $\sqrt{RQ_t}$ | 실효 계수 $\beta_{d,t}$ | 해석 |
|---|---|---|---|
| 조용한 날 | 1 | 0.5 − 0.1 = **0.4** | 어제 RV를 많이 믿음 |
| 요동친 날 | 4 | 0.5 − 0.4 = **0.1** | 어제 RV를 거의 안 믿고 주·월 평균에 의존 |

**왜 RQ가 아니라 √RQ인가:** RQ는 4제곱이라 극단값이 엄청나게 커. 제곱근을 씌우면 규모가 RV와 비슷해지고 이상치의 영향도 줄어서 추정이 안정적이야. 실무 팁 하나 더: $\sqrt{RQ_t}$에서 **표본 평균을 빼고** 넣으면, $\beta_d$를 "평균적인 날의 계수"로 해석할 수 있어서 결과 보고가 깔끔해져.

**왜 일별 항에만 넣나:** 6단계 끝에서 말했듯 주·월 평균은 오차가 이미 평균으로 희석돼 있어서 효과가 작아. 논문에는 세 항 모두에 Q를 넣은 **HARQ-F**(Full) 버전도 있지만, 기본형은 일별 항에만 넣어.

**결과:** BPQ(2016)는 HARQ가 HAR보다 **표본 외 예측력이 일관되게 좋다**고 보고했어. 모형이 거의 그대로이고 변수 하나만 추가했는데 개선된다는 게 이 논문이 널리 쓰이는 이유야.

---

## 9단계: 네 프로젝트에 쓸 때의 함정

**RQ 자체가 RV보다 훨씬 부정확해.** 4제곱은 봉 하나에 극도로 민감해서, N = 39면 RQ는 RV보다 훨씬 노이즈가 커. "오차를 추정하는 지표에 또 오차가 있는" 상황이야. RSkew 때처럼 일별 값이 크게 튀니, 이상치 처리가 중요해.

**점프가 RQ를 폭발시켜.** 점프 봉은 4제곱되면서 RQ를 압도해. 점프에 강건한 대안(인접 봉들의 절댓값을 곱해 만드는 tri-power / quad-power quarticity)이 있어. 원리는 BV와 똑같이 "자기 자신 대신 이웃과 곱하기"야.

**비현실적인 예측값.** RQ가 극단적으로 크면 실효 계수가 음수가 되고, 예측 RV가 음수로 나올 수 있어. 분산은 음수일 수 없으니, 예측값을 합리적인 범위로 제한하는 필터가 필요할 수 있어.

**오버나이트 빠짐.** 장중 RV는 오버나이트 변동을 포함하지 않아. 한국 개별 종목은 오버나이트 변동의 비중이 커서, 네가 예측하는 게 "하루 전체 변동성"인지 "장중 변동성"인지 명확히 정의해야 해.

**표준오차 계산.** 주·월 평균은 겹치는 기간으로 만든 변수라 회귀 잔차에 자기상관이 생겨. 일반 OLS 표준오차는 틀리니 **Newey-West(HAC) 표준오차**를 써야 해.

**평가 지표.** 변동성 예측은 MSE와 함께 **QLIKE** 손실함수를 많이 써.

$$
QLIKE = \frac{RV_{t+1}}{\widehat{RV}_{t+1}} - \ln\frac{RV_{t+1}}{\widehat{RV}_{t+1}} - 1
$$

MSE는 고변동 날 몇 개에 결과가 좌우되지만, QLIKE는 비율로 평가해서 그 문제가 덜하고, 측정오차가 있는 RV를 정답으로 써도 모형 순위가 왜곡되지 않는 성질이 있어.

---

## 계산 코드

```python
import numpy as np
import pandas as pd
import statsmodels.api as sm

def rv_rq(r: pd.Series) -> pd.Series:
    r = r.dropna().values
    N = len(r)
    return pd.Series({'RV': np.sum(r**2),
                      'RQ': (N / 3) * np.sum(r**4)})

daily = df.groupby('date')['ret10m'].apply(rv_rq).unstack().sort_index()

daily['RV_w'] = daily['RV'].rolling(5).mean()
daily['RV_m'] = daily['RV'].rolling(22).mean()
sqrtRQ = np.sqrt(daily['RQ'])
daily['RQ_int'] = (sqrtRQ - sqrtRQ.mean()) * daily['RV']   # 평균 제거 후 상호작용
daily['y'] = daily['RV'].shift(-1)                          # 내일 RV
d = daily.dropna()

X_har  = sm.add_constant(d[['RV', 'RV_w', 'RV_m']])
X_harq = sm.add_constant(d[['RV', 'RQ_int', 'RV_w', 'RV_m']])

har  = sm.OLS(d['y'], X_har).fit(cov_type='HAC', cov_kwds={'maxlags': 22})
harq = sm.OLS(d['y'], X_harq).fit(cov_type='HAC', cov_kwds={'maxlags': 22})
print(harq.summary())   # RQ_int 계수가 음수인지 확인
```

(평균 제거에 전체 표본 평균을 쓰면 엄밀히는 미래 정보가 섞여. 표본 외 평가를 할 때는 롤링 윈도우 안의 평균만 써야 해.)

---

## 네 프로젝트와의 연결

HARQ에는 네 레짐 연구와 연결되는 중요한 관점이 있어. **HARQ는 일종의 "연속적인 레짐 모형"**이야.

- Markov switching이나 HMM(8, 9번)은 "지금은 레짐 1, 지금은 레짐 2"처럼 **이산적으로** 계수를 바꿔.
- HARQ는 $\sqrt{RQ_t}$라는 관측 가능한 상태변수에 따라 계수를 **연속적으로** 바꿔.

그러니 네가 "레짐별로 변동성 동학이 다르다"를 주장하고 싶다면, HARQ는 반드시 이겨야 할 벤치마크야. **"레짐 전환 HAR이 HARQ보다 표본 외 예측력이 좋은가?"**라는 비교 자체가 좋은 연구 질문이야. 레짐 모형이 HARQ도 못 이긴다면, 복잡한 레짐 구조가 사실 "측정오차가 큰 날과 작은 날"을 구분하고 있었을 뿐일 수도 있어.

앞에서 배운 semivariance 버전(SHAR: $RV_t$를 $RS_t^+$, $RS_t^-$로 쪼갠 HAR)과 결합한 모형도 이 분야에서 자주 쓰는 확장이야.

---

**한 줄 요약:** HAR은 일·주·월 평균 RV 세 개로 내일 RV를 예측하는 간단하면서 강력한 모형이야. 그런데 일별 RV에는 σ⁴에 비례하는 측정오차가 있어서(10분봉이면 약 23%), 오차가 큰 날에도 같은 가중치를 주는 게 문제야. HARQ는 realized quarticity($RQ$)로 그날의 측정 정확도를 추정해서, **부정확한 날에는 어제 RV의 가중치를 자동으로 낮추는** 모형이야.

다음 8번 **HMM**은 드디어 레짐을 직접 다루는 첫 번째 개념이야. 준비되면 말해줘.